# ETL Silver - Lluvia diaria ANA

Calcula lluvia acumulada diaria por estacion. Regla R8 (Decision 019): sin umbral de
exclusion -- publica toda estacion con dato real. La calidad medida aca es informativa
(attribute_quality), no bloquea ni borra publicaciones; la cobertura real por sub-cuenca
se expone como columna en Gold.

In [ ]:
from datetime import date, timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F

BRONZE_TABLE = 'weather.bronze.ana_rio_uruguai'
TARGET_TABLE = 'weather.silver.rainfall_daily'
QUALITY_TABLE = 'weather.silver.attribute_quality'
THRESHOLD_PCT = 0.90
QUALITY_WINDOW_DAYS = 30

try:
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental'])
    dbutils.widgets.text('incremental_lookback_days', '14')
    load_mode = dbutils.widgets.get('load_mode')
    incremental_lookback_days = int(dbutils.widgets.get('incremental_lookback_days'))
except Exception:
    load_mode = 'incremental'
    incremental_lookback_days = 14

print(f'load_mode={load_mode}, incremental_lookback_days={incremental_lookback_days}')

In [ ]:
def parse_decimal(column_name):
    return F.regexp_replace(F.trim(F.col(column_name).cast('string')), ',', '.').cast('double')


def normalize_bronze(df):
    return (
        df.select('codigoestacao', 'Data_Hora_Medicao', 'Chuva_Adotada')
        .withColumn('codigoestacao', F.col('codigoestacao').cast('string'))
        .withColumn('medicao_ts', F.to_timestamp('Data_Hora_Medicao'))
        .withColumn('fecha', F.to_date('medicao_ts'))
        .withColumn('lluvia_mm', parse_decimal('Chuva_Adotada'))
        .filter(F.col('codigoestacao').isNotNull())
        .filter(F.col('fecha').isNotNull())
        .filter((F.col('lluvia_mm').isNull()) | (F.col('lluvia_mm') >= F.lit(0.0)))
    )


def aggregate_daily(df):
    return (
        df.groupBy('fecha', 'codigoestacao')
        .agg(
            F.sum('lluvia_mm').alias('lluvia_acumulada_mm'),
            F.count('*').cast('bigint').alias('registros_total'),
            F.count('lluvia_mm').cast('bigint').alias('registros_validos'),
            F.min('medicao_ts').alias('first_medicao_ts'),
            F.max('medicao_ts').alias('last_medicao_ts'),
        )
        .withColumn('source_table', F.lit(BRONZE_TABLE))
        .withColumn('processed_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
        .select('fecha', 'codigoestacao', 'lluvia_acumulada_mm', 'registros_total', 'registros_validos', 'first_medicao_ts', 'last_medicao_ts', 'source_table', 'processed_at', 'updated_at')
    )


def apply_incremental_window(df):
    if load_mode == 'full':
        return df

    max_target_fecha = spark.table(TARGET_TABLE).agg(F.max('fecha').alias('max_fecha')).first()['max_fecha']
    if max_target_fecha is None:
        return df

    start_date = max_target_fecha - timedelta(days=incremental_lookback_days)
    print(f'Processing Silver rainfall from {start_date}')
    return df.filter(F.col('fecha') >= F.lit(start_date))


def apply_quality_window(df):
    # Evalua calidad solo sobre una ventana reciente: el historico de ana_rio_uruguai
    # tiene un backlog masivo de dias nunca ingeridos (y algun timestamp corrupto muy
    # antiguo) que deja el missing_pct siempre por encima del umbral si se mide contra
    # todo el rango. Con esto, la ingesta reciente puede habilitar la publicacion.
    end_date = date.today() - timedelta(days=1)
    start_date = end_date - timedelta(days=QUALITY_WINDOW_DAYS - 1)
    return df.filter((F.col('fecha') >= F.lit(start_date)) & (F.col('fecha') <= F.lit(end_date)))


def build_quality(df):
    return (
        df.agg(
            F.min('fecha').alias('evaluation_start_date'),
            F.max('fecha').alias('evaluation_end_date'),
            F.countDistinct(F.when(F.col('lluvia_acumulada_mm').isNotNull(), F.col('fecha'))).cast('bigint').alias('observed_days'),
        )
        .withColumn('expected_days', F.when(F.col('evaluation_start_date').isNull(), F.lit(0)).otherwise(F.datediff(F.col('evaluation_end_date'), F.col('evaluation_start_date')) + F.lit(1)).cast('bigint'))
        .withColumn('missing_days', F.greatest(F.col('expected_days') - F.col('observed_days'), F.lit(0)).cast('bigint'))
        .withColumn('missing_pct', F.when(F.col('expected_days') == 0, F.lit(1.0)).otherwise(F.col('missing_days') / F.col('expected_days')))
        .withColumn('threshold_pct', F.lit(THRESHOLD_PCT))
        .withColumn('is_usable', F.col('missing_pct') <= F.col('threshold_pct'))
        .withColumn('source_layer', F.lit('silver'))
        .withColumn('source_table', F.lit(TARGET_TABLE))
        .withColumn('source_name', F.lit('rainfall_daily'))
        .withColumn('attribute_name', F.lit('lluvia_acumulada_mm'))
        .withColumn('grain', F.lit('global_source_daily'))
        .withColumn('evaluated_at', F.current_timestamp())
        .withColumn('notes', F.lit(f'Lluvia ANA global; ventana de calidad de {QUALITY_WINDOW_DAYS} dias; metrica informativa (R8, Decision 019) -- ya no bloquea publicacion'))
        .withColumn('created_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
        .select('source_layer', 'source_table', 'source_name', 'attribute_name', 'grain', 'evaluation_start_date', 'evaluation_end_date', 'expected_days', 'observed_days', 'missing_days', 'missing_pct', 'threshold_pct', 'is_usable', 'evaluated_at', 'notes', 'created_at', 'updated_at')
    )


def merge_quality(quality_df):
    DeltaTable.forName(spark, QUALITY_TABLE).alias('t').merge(
        quality_df.alias('s'),
        't.source_table = s.source_table AND t.attribute_name = s.attribute_name AND t.grain = s.grain',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


def merge_daily(daily_df):
    if daily_df.limit(1).count() == 0:
        print('No rainfall rows to merge')
        return

    DeltaTable.forName(spark, TARGET_TABLE).alias('t').merge(
        daily_df.alias('s'),
        't.fecha = s.fecha AND t.codigoestacao = s.codigoestacao',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [ ]:
bronze_all = normalize_bronze(spark.table(BRONZE_TABLE))
daily_all_for_quality = aggregate_daily(bronze_all)

# R8 (Decision 019): sin umbral de exclusion para lluvia. La calidad se mide y se
# publica en attribute_quality solo a fines informativos -- ya no bloquea publicacion
# ni borra filas. Antes, un DELETE global castigaba a todas las estaciones si el
# promedio de la red caia por debajo del umbral, aunque hubiera estaciones con serie
# excelente.
quality_df = build_quality(apply_quality_window(daily_all_for_quality))
merge_quality(quality_df)

quality_row = quality_df.select('missing_pct', 'is_usable').first()
print(f"rainfall missing_pct={quality_row['missing_pct']} (informativo, no bloquea publicacion)")

daily_to_merge = aggregate_daily(apply_incremental_window(bronze_all))
merge_daily(daily_to_merge)

spark.table(QUALITY_TABLE).filter((F.col('source_table') == F.lit(TARGET_TABLE)) & (F.col('attribute_name') == F.lit('lluvia_acumulada_mm'))).show(truncate=False)
spark.table(TARGET_TABLE).agg(F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'), F.count('*').alias('rows'), F.countDistinct('codigoestacao').alias('estaciones')).show()